# 1e. Blank dimensions

About 30 minutes. Slow down here.

A blank dimension is an axis of length 1. It holds no data. You add one to
control **which axes line up** when two tensors meet.
> **Solution.** Read it after your own attempt has failed, then close it and retype the answer from memory.


In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False, linewidth=110)


class _Todo:
    def __repr__(self):
        return "<fill me in>"


TODO = _Todo()


def S(t):
    """Shape as a tuple. Type this constantly."""
    return tuple(t.shape)


def check(label, got, want=None, shape=None, dtype=None, atol=1e-5, rtol=1e-4):
    """Check one answer. Prints what went wrong, then raises."""
    bad = []
    if isinstance(got, _Todo):
        bad.append("not filled in yet")
    else:
        if shape is not None:
            if not torch.is_tensor(got):
                bad.append(f"expected a tensor, got {type(got).__name__}")
            elif S(got) != tuple(shape):
                bad.append(f"shape is {S(got)}, want {tuple(shape)}")
        if dtype is not None and torch.is_tensor(got) and got.dtype != dtype:
            bad.append(f"dtype is {got.dtype}, want {dtype}")
        if want is not None:
            g = got if torch.is_tensor(got) else torch.as_tensor(got)
            w = want if torch.is_tensor(want) else torch.as_tensor(want)
            if S(g) != S(w):
                bad.append(f"shape is {S(g)}, want {S(w)}")
            elif not torch.allclose(g.double(), w.double(), atol=atol, rtol=rtol):
                bad.append(f"values differ, largest gap "
                           f"{(g.double() - w.double()).abs().max().item():.3e}")
    if bad:
        print(f"FAIL  {label}")
        for b in bad:
            print(f"      {b}")
        raise AssertionError(label)
    print(f"ok    {label}")


def same(label, a, b):
    """Two shapes or tuples must match."""
    if isinstance(a, _Todo):
        print(f"FAIL  {label}: not filled in yet")
        raise AssertionError(label)
    if tuple(a) != tuple(b):
        print(f"FAIL  {label}\n      you said {tuple(a)}, it is {tuple(b)}")
        raise AssertionError(label)
    print(f"ok    {label} = {tuple(a)}")

## Four ways to add one

In [ ]:
v = torch.tensor([1.0, 2.0, 3.0, 4.0])

print("v            ", S(v))
print("v[None]      ", S(v[None]), "   front")
print("v[:, None]   ", S(v[:, None]), "   after axis 0 - a column")
print("v[..., None] ", S(v[..., None]), "   end")
print("v[None, None]", S(v[None, None]), "   two at once")

`unsqueeze(k)` is the same as putting `None` at position `k`. Use whichever
reads better.

## What it buys you

In [ ]:
print("v * v                  ", S(v * v), "  elementwise")
print("v[None, :] * v[:, None]", S(v[None, :] * v[:, None]), "  outer product")

A row times a column is an outer product. RoPE builds its frequency table
with exactly that, and you will write it in **1m**.

## squeeze removes one, conditionally

In [ ]:
t = torch.zeros(1, 5, 1)

print("t            ", S(t))
print("t.squeeze(0) ", S(t.squeeze(0)), "  removed")
print("t.squeeze(1) ", S(t.squeeze(1)), "  length 5, did nothing")
print("t.squeeze()  ", S(t.squeeze()), "     removed both")

Always pass the axis. Bare `squeeze()` eats every length-1 axis, including
your batch axis the day batch size is 1.

## Insertions renumber everything to their right

In [ ]:
z = torch.zeros(2, 3, 4, 5)

print("z           ", S(z), "  axis 2 has length", z.shape[2])
print("z[:, None]  ", S(z[:, None]), " the old axis 2 is now axis 3")
print("z.sum(1)    ", S(z.sum(1)), "    the old axis 2 is now axis 1")

Most confusion about blank dimensions is really this. Every insertion and
every reduction shifts the axes after it.

## Drill 1 (easy)

Make `small` line up with `big` on the named axes. Use `None` indexing,
`unsqueeze` or `squeeze` only. No `expand`, no `repeat`, no `reshape`.

| | `big` | `small` | line up on |
|---|---|---|---|
| a | `(B, H, T, DH)` | `(T, DH)` | `T, DH` |
| b | `(B, H, T, DH)` | `(H,)` | `H` |
| c | `(K, N)` | `(K,)` | `K` |
| d | `(K, N)` | `(N,)` | `N` |
| e | `(T, DH)` | `(1, T, DH)` | one axis too many |

In [ ]:
big = {"a": torch.randn(2, 4, 3, 8), "b": torch.randn(2, 4, 3, 8),
       "c": torch.randn(6, 16), "d": torch.randn(6, 16), "e": torch.randn(3, 8)}
small = {"a": torch.randn(3, 8), "b": torch.randn(4), "c": torch.randn(6),
         "d": torch.randn(16), "e": torch.randn(1, 3, 8)}

ans = {"a": small["a"][None, None],
       "b": small["b"][None, :, None, None],
       "c": small["c"][:, None],
       "d": small["d"][None, :],
       "e": small["e"].squeeze(0)}

check("a lines up on (T,DH)", (big["a"] * ans["a"])[0, 0],
      want=big["a"][0, 0] * small["a"])
check("b lines up on H", (big["b"] * ans["b"])[0, 2],
      want=big["b"][0, 2] * small["b"][2])
check("c lines up on K", (big["c"] * ans["c"])[3], want=big["c"][3] * small["c"][3])
check("d lines up on N", (big["d"] * ans["d"])[:, 3],
      want=big["d"][:, 3] * small["d"][3])
check("e", big["e"] * ans["e"], shape=(3, 8))
check("e dropped the axis", torch.tensor(ans["e"].ndim), want=2)

## expand, repeat, repeat_interleave

All three make an axis longer. They differ in cost and, more importantly, in
the **order** of the result.

In [ ]:
kv = torch.tensor([10.0, 20.0])

print("kv                     ", kv.tolist())
print("kv.repeat(2)           ", kv.repeat(2).tolist(), "  a b a b")
print("kv.repeat_interleave(2)", kv.repeat_interleave(2).tolist(), "  a a b b")

`expand` works only on a length-1 axis and gives a view with stride 0: the
same memory read twice. Free, and read-only.

In [ ]:
e = kv[:, None].expand(2, 2)

print("shape", S(e), " stride", e.stride(), " shares memory:", e.data_ptr() == kv.data_ptr())
try:
    e[0, 0] = 1.0
except RuntimeError as err:
    print("writing to it ->", str(err).split(".")[0])

## Drill 2 (medium)

BioMistral has 8 key/value heads serving 32 query heads. **Key/value head `j`
serves query heads `4j` to `4j+3`.**

Write `repeat_kv` so that `x` of shape `(B, HKV, T, DH)` becomes
`(B, HKV*n_rep, T, DH)` with that mapping, copying the data at most once.
It must also work when `n_rep == 1`.

In [ ]:
def repeat_kv(x, n_rep):
    b, hkv, t, dh = x.shape
    if n_rep == 1:
        return x
    return x[:, :, None].expand(b, hkv, n_rep, t, dh).reshape(b, hkv * n_rep, t, dh)

Stuck? Two operations get you there and the first one is free. Uncomment the
hint only after trying.

In [ ]:
# HINT: insert a blank axis after the head axis, expand it to n_rep, then
# HINT: collapse the two head axes together.

In [ ]:
HKV, G, T, DH = 2, 2, 3, 2
xk = torch.randn(1, HKV, T, DH)
rk = repeat_kv(xk, G)

check("shape", rk, shape=(1, HKV * G, T, DH))
for j in range(HKV):
    for r in range(G):
        check(f"kv head {j} -> q head {j*G+r}", rk[:, j * G + r], want=xk[:, j])
check("n_rep=1 is the identity", repeat_kv(xk, 1), want=xk)

A `repeat`-based answer has the right shape and the wrong contents. It gives
`kv0, kv1, kv0, kv1` where you need `kv0, kv0, kv1, kv1`.

Nothing crashes. The model produces slightly wrong text. This has bitten
everyone who has written this.

## Predict

For each expression, answer `"ok"` or `"raise"`. Three of them raise.

In [ ]:
p = {"a": "ok", "b": "raise", "c": "ok", "d": "raise", "e": "ok", "f": "raise"}

cases = {
    "a": "torch.zeros(4, 3) * torch.zeros(3)",
    "b": "torch.zeros(4, 3) * torch.zeros(4)",
    "c": "torch.zeros(4, 1).expand(4, 7)",
    "d": "torch.zeros(4, 2).expand(4, 7)",
    "e": "torch.zeros(1, 5, 1).squeeze(1)",
    "f": "torch.zeros(4, 1).expand(4, 7).mul_(2.0)",
}
for key, expr in cases.items():
    try:
        eval(expr)
        real = "ok"
    except RuntimeError:
        real = "raise"
    assert not isinstance(p[key], _Todo), f"{key}: not filled in"
    assert p[key] == real, f"FAIL {key}  {expr}\n  you said {p[key]!r}, it is {real!r}"
    print(f"ok    {key}  {expr}  ->  {real}")

`expand` cannot grow a length-2 axis, and you cannot write into a stride-0
view.

## Next

**1f** — the rule that makes blank dimensions do their work.